In [1]:
import sys 
sys.path.append("..") 

import numpy as np
import pandas as pd
from SWMM import SWMM_ENV
from DQN import Buffer
from DQN import DQN as DQN
import tensorflow as tf
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import os

e:\anaconda3\envs\tensorflow-cpu\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
Train=True
init_train=True

tf.compat.v1.reset_default_graph()

env_params={
        'orf':os.path.dirname(os.getcwd())+'/SWMM/chaohu',
        'orf_save':'chaohu_RTC',
        'parm':os.path.dirname(os.getcwd())+'/states_yaml/chaohu',
        'advance_seconds':300,
        'kf':1,
        'kc':1,
        'reward_type':'1'
    }
env=SWMM_ENV.SWMM_ENV(env_params)

raindata = np.load(os.path.dirname(os.getcwd())+'/rainfall/training_raindata.npy').tolist()


agent_params={
    'state_dim':len(env.config['states']),
    'action_dim':2**len(env.config['action_assets']),

    'encoding_layer':[100,100],
    'value_layer':[100,200],
    'advantage_layer':[100,100],
    'num_rain':50,

    'train_iterations':20,
    'training_step':1000,
    'gamma':0.3,
    'epsilon':1,
    'ep_min':1e-100,
    'ep_decay':0.999,
    'learning_rate':0.01,

    'action_table':pd.read_csv(os.path.dirname(os.getcwd())+'/SWMM/action_table.csv').values[:,1:],
}


model = DQN.DQN(agent_params)
if init_train:
    model.model.save_weights('./model/dqn.h5')    
    model.target_model.save_weights('./model/target_dqn.h5')    
model.load_model('./model/')

In [15]:
def sample_action(observation,model,train_log):
    #input state, output action
    if train_log:
        #epsilon greedy
        pa = np.random.uniform()
        if model.params['epsilon'] < pa:
            print('################1')
            action_value = model.model(observation)
            action = tf.argmax(action_value)
            #action = tf.squeeze(tf.random.categorical(action_value, 1), axis=1)
        else:
            print('################2')
            logits = tf.compat.v1.random_normal([1, 128], mean=0, stddev=1)
            action = tf.squeeze(tf.random.categorical(logits, 1), axis=1)
    else:
        action_value = model.model(observation)
        action = tf.argmax(action_value)
        #action = tf.squeeze(tf.random.categorical(action_value, 1), axis=1)
    return action

In [4]:
def interact(i,ep):
    env=SWMM_ENV.SWMM_ENV(env_params)
    tem_model = DQN.DQN(agent_params)
    tem_model.load_model('./model/')
    tem_model.params['epsilon']=ep
    s,a,r,s_ = [],[],[],[]
    observation, episode_return, episode_length = env.reset(raindata[i],i,True,os.path.dirname(os.getcwd())), 0, 0
    
    done = False
    while not done:
        # Get the action, and take one step in the environment
        observation = np.array(observation).reshape(1, -1)
        action = DQN.sample_action(observation,tem_model,True)
        at = tem_model.action_table[int(action[0].numpy())].tolist()
        observation_new, reward, results, done = env.step(at)
        episode_return += reward
        episode_length += 1

        # Store obs, act, rew
        # buffer.store(observation, action, reward, value_t, logprobability_t)
        s.append(observation)
        a.append(action)
        r.append(reward)
        s_.append(observation_new)
        
        # Update the observation
        observation = observation_new
    # Finish trajectory if reached to a terminal state
    last_value = 0 if done else tem_model.predict(observation.reshape(1, -1))
    return s,a,r,s_,last_value,episode_return,episode_length

In [5]:
res = interact(0,0.01)

0
63
0
0
0
0
0
0
0
0
0
0
0
0
0
0
6
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
